# Student Performance & At-Risk Prediction
## ML Pipeline

## Section 02 — Library Imports

In [1]:
# Standard library
import json
import warnings
from datetime import datetime
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Scikit-learn — preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Scikit-learn — models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Scikit-learn — model selection & evaluation
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
)

# Model persistence
import joblib

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# Reproducibility
random_seed = 42
np.random.seed(random_seed)
warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")
print(f"Random seed : {random_seed}")
print(f"Pandas      : {pd.__version__}")
print(f"NumPy       : {np.__version__}")

Libraries loaded successfully.
Random seed : 42
Pandas      : 2.1.4
NumPy       : 1.26.4


In [2]:
# Define all file paths in one place.
# Every subsequent cell reads from these variables — never hardcodes a path.

project_root  = Path("d:/student-performance-risk-prediction")
data_path     = project_root / "data" / "generated" / "student_performance.csv"
artifacts_dir = project_root / "artifacts"
figures_dir   = project_root / "reports" / "figures"

# Create output directories if they don't already exist
artifacts_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("Project root   :", project_root)
print("Data file      :", data_path)
print("Data file exists:", data_path.exists())
print("Artifacts dir  :", artifacts_dir)
print("Figures dir    :", figures_dir)

Project root   : d:\student-performance-risk-prediction
Data file      : d:\student-performance-risk-prediction\data\generated\student_performance.csv
Data file exists: True
Artifacts dir  : d:\student-performance-risk-prediction\artifacts
Figures dir    : d:\student-performance-risk-prediction\reports\figures


## Section 03 — Dataset Loading

In [3]:
# Load the CSV generated in Day 1.
# parse_dates converts the prediction_date string column to datetime immediately.

df = pd.read_csv(data_path, parse_dates=["prediction_date"])

print("Shape           :", df.shape)
print("Date range      :", df["prediction_date"].min().date(),
      "to", df["prediction_date"].max().date())
print("\nColumn names:")
print(df.columns.tolist())

Shape           : (5000, 19)
Date range      : 2025-01-01 to 2025-10-31

Column names:
['student_id', 'cohort_id', 'prediction_date', 'attendance_rate', 'recent_attendance_rate', 'assignments_due', 'assignments_completed', 'assignment_completion_rate', 'missing_assignments', 'late_submissions', 'late_submission_rate', 'days_since_last_submission', 'previous_score_1', 'previous_score_2', 'previous_score_3', 'average_previous_score', 'score_trend', 'next_assessment_score', 'at_risk_next_assessment']


In [4]:
# Quick look at the first five rows
display(df.head())

,student_id,cohort_id,prediction_date,attendance_rate,recent_attendance_rate,assignments_due,assignments_completed,assignment_completion_rate,missing_assignments,late_submissions,late_submission_rate,days_since_last_submission,previous_score_1,previous_score_2,previous_score_3,average_previous_score,score_trend,next_assessment_score,at_risk_next_assessment
0,STU00001,2025Q1,2025-01-28,65.595820,62.323629,19,10,52.631579,9,3,30.000000,12.0,31.109050,40.545986,49.064901,40.239979,17.955851,52.302733,0
1,STU00002,2025Q3,2025-08-24,59.001961,55.013889,23,13,56.521739,10,6,46.153846,16.0,54.441838,44.553046,38.686174,45.893686,-15.755664,45.676736,0
2,STU00003,2025Q3,2025-07-18,53.932610,55.796970,20,10,50.000000,10,6,60.000000,10.0,55.557314,44.543327,44.151775,48.084139,-11.405540,60.272588,0
3,STU00004,2025Q2,2025-05-14,55.484107,65.226436,25,14,56.000000,11,3,21.428571,15.0,54.629539,50.439654,48.052615,51.040603,-6.576924,67.491728,0
4,STU00005,2025Q2,2025-05-12,81.939543,72.299291,22,10,45.454545,12,2,20.000000,14.0,49.394440,47.326112,57.129738,51.283430,7.735298,42.560398,0


In [5]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   student_id                  5000 non-null   object        
 1   cohort_id                   5000 non-null   object        
 2   prediction_date             5000 non-null   datetime64[ns]
 3   attendance_rate             4912 non-null   float64       
 4   recent_attendance_rate      4924 non-null   float64       
 5   assignments_due             5000 non-null   int64         
 6   assignments_completed       5000 non-null   int64         
 7   assignment_completion_rate  5000 non-null   float64       
 8   missing_assignments         5000 non-null   int64         
 9   late_submissions            5000 non-null   int64         
 10  late_submission_rate        4905 non-null   float64       
 11  days_since_last_submission  4946 non-null   float64     

## Section 04 — Data Quality Analysis

In [6]:
# Knowing which columns have missing values tells us where to apply imputation.
missing_counts = df.isnull().sum()
missing_pct    = (df.isnull().mean() * 100).round(2)
missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing Pct"  : missing_pct,
}).sort_values("Missing Count", ascending=False)

missing_summary = missing_summary[missing_summary["Missing Count"] > 0]
display(missing_summary)

,Missing Count,Missing Pct
late_submission_rate,95,1.90
attendance_rate,88,1.76
recent_attendance_rate,76,1.52
days_since_last_submission,54,1.08


In [7]:
# Target class distribution
# This tells us how imbalanced the problem is.
# Class imbalance means accuracy is a poor metric — a model that
# predicts 'not at risk' for everyone achieves 94.5% accuracy but misses
# every real at-risk student.

target_counts = df["at_risk_next_assessment"].value_counts().rename({0: "Not At-Risk (0)", 1: "At-Risk (1)"})
target_pct    = (df["at_risk_next_assessment"].value_counts(normalize=True) * 100).rename({0: "Not At-Risk (0)", 1: "At-Risk (1)"})

class_summary = pd.DataFrame({"Count": target_counts, "Percentage": target_pct.round(2)})
display(class_summary)

print(f"\nImbalance ratio  : {target_counts['Not At-Risk (0)'] / target_counts['At-Risk (1)']:.1f}:1  (not-at-risk : at-risk)")

,Count,Percentage
at_risk_next_assessment,,
Not At-Risk (0),4725,94.5
At-Risk (1),275,5.5



Imbalance ratio  : 17.2:1  (not-at-risk : at-risk)


In [8]:
# Descriptive statistics for numerical features
# Useful for spotting outliers and understanding feature ranges.

numerical_cols = df.select_dtypes(include="number").columns.tolist()
display(df[numerical_cols].describe().round(2))

,attendance_rate,recent_attendance_rate,assignments_due,assignments_completed,assignment_completion_rate,missing_assignments,late_submissions,late_submission_rate,days_since_last_submission,previous_score_1,previous_score_2,previous_score_3,average_previous_score,score_trend,next_assessment_score,at_risk_next_assessment
count,4912.00,4924.00,5000.00,5000.00,5000.00,5000.00,5000.00,4905.00,4946.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,68.74,68.59,20.01,12.47,62.24,7.54,3.55,29.93,9.40,53.10,52.91,52.64,52.88,-0.46,60.35,0.06
std,12.93,15.16,3.20,3.95,16.77,3.53,1.81,15.37,4.19,11.64,12.80,14.60,12.16,9.37,12.88,0.23
min,29.10,16.67,15.00,2.00,9.09,0.00,0.00,0.00,0.00,18.34,15.57,6.92,20.09,-40.10,18.57,0.00
25%,59.23,58.18,17.00,10.00,50.00,5.00,2.00,18.75,7.00,44.79,43.66,42.25,43.90,-6.82,51.08,0.00
50%,68.55,68.25,20.00,12.00,62.50,7.00,3.00,28.57,9.00,52.99,52.61,52.35,52.55,-0.33,60.32,0.00
75%,78.27,79.52,23.00,15.00,75.00,10.00,5.00,38.89,12.00,61.31,61.83,63.12,61.78,5.83,69.39,0.00
max,100.00,100.00,25.00,25.00,100.00,20.00,11.00,100.00,23.00,91.92,93.07,98.13,88.09,36.53,100.00,1.00


In [9]:
# Duplicate check
n_duplicates = df.duplicated(subset="student_id").sum()
print(f"Duplicate student_id rows : {n_duplicates}")
print(f"All student_ids unique    : {df['student_id'].is_unique}")

Duplicate student_id rows : 0
All student_ids unique    : True


## Section 05 — Feature Engineering

In [10]:
# attendance_drop
# A student whose recent attendance is much lower than their overall rate
# is showing a warning signal.  Positive = improving, Negative = declining.
df["attendance_drop"] = df["attendance_rate"] - df["recent_attendance_rate"]
print("attendance_drop  — sample values:")
print(df["attendance_drop"].describe().round(2))

attendance_drop  — sample values:
count    4840.00
mean        0.15
std         7.87
min       -29.84
25%        -4.95
50%         0.13
75%         5.23
max        31.25
Name: attendance_drop, dtype: float64


In [11]:
# score_volatility
# A student who alternates between very high and very low scores is less
# stable than one with consistent performance.
# std across the three historical scores, computed row-wise (axis=1).
score_cols = ["previous_score_1", "previous_score_2", "previous_score_3"]
df["score_volatility"] = df[score_cols].std(axis=1)
print("score_volatility — sample values:")
print(df["score_volatility"].describe().round(2))

score_volatility — sample values:
count    5000.00
mean        5.18
std         2.75
min         0.10
25%         3.16
50%         4.82
75%         6.86
max        20.05
Name: score_volatility, dtype: float64


In [12]:
# Confirm the new columns are in the DataFrame
print("Columns after feature engineering:")
print(df.columns.tolist())
print("\nShape:", df.shape)

Columns after feature engineering:
['student_id', 'cohort_id', 'prediction_date', 'attendance_rate', 'recent_attendance_rate', 'assignments_due', 'assignments_completed', 'assignment_completion_rate', 'missing_assignments', 'late_submissions', 'late_submission_rate', 'days_since_last_submission', 'previous_score_1', 'previous_score_2', 'previous_score_3', 'average_previous_score', 'score_trend', 'next_assessment_score', 'at_risk_next_assessment', 'attendance_drop', 'score_volatility']

Shape: (5000, 21)


## Section 06 — Feature Selection & Leakage Prevention

In [13]:
# Define the definitive feature list
# Every model will be trained using these features.
# Never silently add or remove columns after this point.

features = [
    # Attendance
    "attendance_rate",
    "recent_attendance_rate",
    "attendance_drop",
    # Assignment behaviour
    "assignment_completion_rate",
    "missing_assignments",
    "late_submission_rate",
    "days_since_last_submission",
    # Historical academic performance
    "previous_score_1",
    "previous_score_2",
    "previous_score_3",
    "average_previous_score",
    "score_trend",
    "score_volatility",
]

target = "at_risk_next_assessment"

# Columns with known missing values that require imputation
features_with_missing = ["attendance_rate","recent_attendance_rate","late_submission_rate","days_since_last_submission",]

# Note: attendance_drop is derived from attendance_rate and recent_attendance_rate
# so it also has missing values in some rows.
features_with_missing.append("attendance_drop")

features_no_missing = [f for f in features if f not in features_with_missing]

print("Total features :", len(features))
print("\nFeatures with missing values:")
print(features_with_missing)
print("\nFeatures with no missing values:")
print(features_no_missing)
print("\nTarget:", target)

# Confirming all features actually exist in the DataFrame
missing_features = [f for f in features if f not in df.columns]
print("\nFeatures requested but missing from df:", missing_features or "None")

Total features : 13

Features with missing values:
['attendance_rate', 'recent_attendance_rate', 'late_submission_rate', 'days_since_last_submission', 'attendance_drop']

Features with no missing values:
['assignment_completion_rate', 'missing_assignments', 'previous_score_1', 'previous_score_2', 'previous_score_3', 'average_previous_score', 'score_trend', 'score_volatility']

Target: at_risk_next_assessment

Features requested but missing from df: None


In [14]:
# Leakage verification
# Explicitly assert that leaky columns are absent from the feature list.

leaky_columns = ["next_assessment_score", target, "student_id","cohort_id", "prediction_date","assignments_due", "assignments_completed", "late_submissions"]
leakage_in_features = [c for c in leaky_columns if c in features]
assert len(leakage_in_features) == 0, f"Leakage detected: {leakage_in_features}"
print("Leakage check passed - no leaky columns in features.")

Leakage check passed - no leaky columns in features.


## Section 07 — Chronological Data Splitting

In [15]:
# Sort the full dataset by prediction_date before splitting.
df = df.sort_values("prediction_date").reset_index(drop=True)

n = len(df)
train_end_idx = int(n * 0.70)
val_end_idx   = int(n * 0.85)

df_train = df.iloc[:train_end_idx].copy()
df_val   = df.iloc[train_end_idx:val_end_idx].copy()
df_test  = df.iloc[val_end_idx:].copy()

print(f"Training set   : {len(df_train):,} rows  "f"| {df_train['prediction_date'].min().date()} to {df_train['prediction_date'].max().date()}")
print(f"Validation set : {len(df_val):,} rows  "f"| {df_val['prediction_date'].min().date()} to {df_val['prediction_date'].max().date()}")
print(f"Test set       : {len(df_test):,} rows  "f"| {df_test['prediction_date'].min().date()} to {df_test['prediction_date'].max().date()}")

# Verify no date overlap between splits
assert df_train["prediction_date"].max() <= df_val["prediction_date"].min(), "Overlap between train and val"
assert df_val["prediction_date"].max() <= df_test["prediction_date"].min(), "Overlap between val and test"


Training set   : 3,500 rows  | 2025-01-01 to 2025-08-03
Validation set : 750 rows  | 2025-08-03 to 2025-09-16
Test set       : 750 rows  | 2025-09-16 to 2025-10-31


In [16]:
# Target distribution per split
# We expect some variation in at-risk rate across splits because the
# synthetic dates are random.  Major deviation would be a data quality concern.

for name, subset in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    at_risk_pct = subset[target].mean() * 100
    print(f"{name:12s} : {len(subset):,} rows | At-Risk = {subset[target].sum()} ({at_risk_pct:.1f}%)")

Train        : 3,500 rows | At-Risk = 200 (5.7%)
Validation   : 750 rows | At-Risk = 35 (4.7%)
Test         : 750 rows | At-Risk = 40 (5.3%)


In [17]:
# Create X / y arrays for each split
x_train = df_train[features].copy()
y_train = df_train[target].copy()

x_val   = df_val[features].copy()
y_val   = df_val[target].copy()

x_test  = df_test[features].copy()
y_test  = df_test[target].copy()

print("x_train shape:", x_train.shape)
print("x_val shape  :", x_val.shape)
print("x_test shape :", x_test.shape)

x_train shape: (3500, 13)
x_val shape  : (750, 13)
x_test shape : (750, 13)


## Section 08 — Preprocessing Pipeline

In [18]:
# Pipeline for features that have missing values: impute then scale.
pipe_with_missing = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# Pipeline for features that have no missing values: scale only.
pipe_no_missing = Pipeline([
    ("scaler", StandardScaler()),
])

# ColumnTransformer applies the right pipeline to the right columns.
preprocessor = ColumnTransformer(
    transformers=[
        ("with_missing", pipe_with_missing, features_with_missing),
        ("no_missing",   pipe_no_missing,   features_no_missing),
    ],
    remainder="drop",  # any column not listed is dropped
)

print("Preprocessor defined.")
print(f"  Columns with imputation + scaling : {len(features_with_missing)}")
print(f"  Columns with scaling only         : {len(features_no_missing)}")
print(f"  Total features                    : {len(features)}")

Preprocessor defined.
  Columns with imputation + scaling : 5
  Columns with scaling only         : 8
  Total features                    : 13


In [19]:
# Sanity check: fit on training data, transform validation
# This confirms the preprocessor works correctly before attaching any model.
# Note: We call fit_transform on train, and only transform on val/test.

x_train_proc = preprocessor.fit_transform(x_train)
x_val_proc   = preprocessor.transform(x_val)
x_test_proc  = preprocessor.transform(x_test)

print("Preprocessed shapes:")
print(f"  x_train_proc : {x_train_proc.shape}")
print(f"  x_val_proc   : {x_val_proc.shape}")
print(f"  x_test_proc  : {x_test_proc.shape}")

# Check no NaN remains after imputation
assert not np.isnan(x_train_proc).any(), "NaN found in training data after preprocessing"
assert not np.isnan(x_val_proc).any(),   "NaN found in validation data after preprocessing"
assert not np.isnan(x_test_proc).any(),  "NaN found in test data after preprocessing"
print("\nNo NaN values remain after preprocessing.")

Preprocessed shapes:
  x_train_proc : (3500, 13)
  x_val_proc   : (750, 13)
  x_test_proc  : (750, 13)

No NaN values remain after preprocessing.


## Section 09 — Baseline Model

In [20]:
# This function is called for every model.  Writing it once avoids repetition
# and ensures we always compute the same metrics in the same way.
def evaluate_model(name, y_true, y_pred, y_proba):
    metrics = {
        "Model"      : name,
        "Accuracy"   : round(accuracy_score(y_true, y_pred), 4),
        "Precision"  : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"     : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1"         : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "ROC-AUC"    : round(roc_auc_score(y_true, y_proba), 4),
        "PR-AUC"     : round(average_precision_score(y_true, y_proba), 4),
    }
    return metrics


In [21]:
# Dummy Classifier
dummy_clf = DummyClassifier(strategy="stratified", random_state=random_seed)
dummy_clf.fit(x_train_proc, y_train)

dummy_pred  = dummy_clf.predict(x_val_proc)
dummy_proba = dummy_clf.predict_proba(x_val_proc)[:, 1]

baseline_metrics = evaluate_model("DummyClassifier", y_val, dummy_pred, dummy_proba)
print("Baseline (Validation Set):")
for k, v in baseline_metrics.items():
    print(f"  {k:<12}: {v}")

print("\nClassification Report:")
print(classification_report(y_val, dummy_pred, target_names=["Not At-Risk", "At-Risk"]))

Baseline (Validation Set):
  Model       : DummyClassifier
  Accuracy    : 0.904
  Precision   : 0.1064
  Recall      : 0.1429
  F1          : 0.122
  ROC-AUC     : 0.5421
  PR-AUC      : 0.0552

Classification Report:
              precision    recall  f1-score   support

 Not At-Risk       0.96      0.94      0.95       715
     At-Risk       0.11      0.14      0.12        35

    accuracy                           0.90       750
   macro avg       0.53      0.54      0.54       750
weighted avg       0.92      0.90      0.91       750



## Section 10 — Initial Model Training

In [22]:
# Logistic Regression
# A linear model.  Requires scaled features (already handled by our pipeline).
# class_weight='balanced' compensates for class imbalance.
# max_iter=1000 avoids convergence warnings on this dataset size.

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(class_weight="balanced",max_iter=1000,random_state=random_seed,)),
])

lr_pipeline.fit(x_train, y_train)
lr_pred  = lr_pipeline.predict(x_val)
lr_proba = lr_pipeline.predict_proba(x_val)[:, 1]

lr_metrics = evaluate_model("LogisticRegression", y_val, lr_pred, lr_proba)
print("Logistic Regression (Validation Set):")
for k, v in lr_metrics.items():
    print(f"  {k:<12}: {v}")

Logistic Regression (Validation Set):
  Model       : LogisticRegression
  Accuracy    : 0.8747
  Precision   : 0.2602
  Recall      : 0.9143
  F1          : 0.4051
  ROC-AUC     : 0.9646
  PR-AUC      : 0.6294


In [23]:
# Random Forest
# An ensemble of decision trees.  More powerful than LR for non-linear
# relationships.  class_weight='balanced' again handles imbalance.
# n_estimators=200 gives enough trees for stable estimates.

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier( n_estimators=200, class_weight="balanced", random_state=random_seed, n_jobs=-1,)),
])
rf_pipeline.fit(x_train, y_train)
rf_pred  = rf_pipeline.predict(x_val)
rf_proba = rf_pipeline.predict_proba(x_val)[:, 1]
rf_metrics = evaluate_model("RandomForest", y_val, rf_pred, rf_proba)
print("Random Forest (Validation Set):")
for k, v in rf_metrics.items():
    print(f"  {k:<12}: {v}")

Random Forest (Validation Set):
  Model       : RandomForest
  Accuracy    : 0.9587
  Precision   : 0.5909
  Recall      : 0.3714
  F1          : 0.4561
  ROC-AUC     : 0.9581
  PR-AUC      : 0.5457


In [24]:
# HistGradientBoosting
# A fast gradient boosting implementation that natively handles missing values
# no imputation strictly needed, but our pipeline still applies it for consistency with other models.
# We pass class weights via the sample_weight mechanism indirectly by using max_iter=300 with early-stop-like configuration.

hgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(max_iter=300,learning_rate=0.05,random_state=random_seed,class_weight="balanced",)),
])

hgb_pipeline.fit(x_train, y_train)

hgb_pred  = hgb_pipeline.predict(x_val)
hgb_proba = hgb_pipeline.predict_proba(x_val)[:, 1]

hgb_metrics = evaluate_model("HistGradientBoosting", y_val, hgb_pred, hgb_proba)
print("HistGradientBoosting (Validation Set):")
for k, v in hgb_metrics.items():
    print(f"  {k:<12}: {v}")

  File "c:\Users\sekar\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\sekar\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sekar\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\sekar\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


HistGradientBoosting (Validation Set):
  Model       : HistGradientBoosting
  Accuracy    : 0.9547
  Precision   : 0.5143
  Recall      : 0.5143
  F1          : 0.5143
  ROC-AUC     : 0.9478
  PR-AUC      : 0.5545


## Section 11 — Initial Model Comparison

In [25]:
# Collecting all initial metrics into a comparison table
all_initial_metrics = [baseline_metrics, lr_metrics, rf_metrics, hgb_metrics]
comparison_df = pd.DataFrame(all_initial_metrics).set_index("Model")
display(comparison_df)

# Highlighting the best value in each column
print("\nBest values per metric:")
for col in comparison_df.columns:
    best_model = comparison_df[col].idxmax()
    best_val   = comparison_df[col].max()
    print(f"  {col:<12}: {best_model} ({best_val})")

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
Model,,,,,,
DummyClassifier,0.9040,0.1064,0.1429,0.1220,0.5421,0.0552
LogisticRegression,0.8747,0.2602,0.9143,0.4051,0.9646,0.6294
RandomForest,0.9587,0.5909,0.3714,0.4561,0.9581,0.5457
HistGradientBoosting,0.9547,0.5143,0.5143,0.5143,0.9478,0.5545



Best values per metric:
  Accuracy    : RandomForest (0.9587)
  Precision   : RandomForest (0.5909)
  Recall      : LogisticRegression (0.9143)
  F1          : HistGradientBoosting (0.5143)
  ROC-AUC     : LogisticRegression (0.9646)
  PR-AUC      : LogisticRegression (0.6294)


In [26]:
# Plotly bar chart: Initial model comparison
metrics_to_plot = ["Recall", "Precision", "F1", "ROC-AUC", "PR-AUC"]

fig_comparison = go.Figure()
for metric in metrics_to_plot:
    fig_comparison.add_trace(go.Bar(
        name=metric,
        x=comparison_df.index.tolist(),
        y=comparison_df[metric].values,
        text=[f"{v:.3f}" for v in comparison_df[metric].values],
        textposition="outside",
    ))

fig_comparison.update_layout(
    title="Initial Model Comparison — Validation Set",
    barmode="group",
    xaxis_title="Model",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1.15]),
    template="plotly_white",
    legend_title="Metric",
)
fig_comparison.show()

# Save for Streamlit
fig_comparison.write_json(str(figures_dir / "initial_model_comparison.json"))
fig_comparison.write_html(str(figures_dir / "initial_model_comparison.html"))
print("Saved: initial_model_comparison.json & .html")

Saved: initial_model_comparison.json & .html


In [27]:
# Confusion matrices for all initial models
# True Positive  (TP): Model says at-risk, student is at-risk - Correct alert
# True Negative  (TN): Model says not at-risk, student is not - Correct dismissal
# False Positive (FP): Model says at-risk, student is not - Unnecessary alert
# False Negative (FN): Model says not at-risk, student is at-risk - Missed student

# In this problem, FN is the most costly error.
# A missed at-risk student receives no support and may fail.

for name, pred in [("LogisticRegression", lr_pred),
                   ("RandomForest",        rf_pred),
                   ("HistGradientBoosting", hgb_pred)]:
    cm = confusion_matrix(y_val, pred)
    print(f"\n{name}")
    print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"  FN={cm[1,0]}  TP={cm[1,1]}")
    print(f"  False Negatives (missed at-risk students): {cm[1,0]}")


LogisticRegression
  TN=624  FP=91
  FN=3  TP=32
  False Negatives (missed at-risk students): 3

RandomForest
  TN=706  FP=9
  FN=22  TP=13
  False Negatives (missed at-risk students): 22

HistGradientBoosting
  TN=698  FP=17
  FN=17  TP=18
  False Negatives (missed at-risk students): 17


## Section 12 — Hyperparameter Optimisation

In [28]:
# TimeSeriesSplit setup
# 5 folds within the training set, ordered by time.
tscv = TimeSeriesSplit(n_splits=5)


In [29]:
# Tune Logistic Regression
# C: inverse regularisation strength. Small C = more regularisation.
# solver: algorithm used to fit the model. 'saga' supports l1 penalty.
# penalty: l1 produces sparse models (some coefficients go to zero).

lr_param_grid = {
    "model__C"           : [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver"      : ["saga", "lbfgs"],
    "model__penalty"     : ["l2"],
    "model__class_weight": ["balanced"],
    "model__max_iter"    : [1000],
}

lr_search = RandomizedSearchCV(
    estimator=lr_pipeline,
    param_distributions=lr_param_grid,
    n_iter=12,
    cv=tscv,
    scoring="average_precision",   # PR-AUC
    refit=True,                    # re-fit best model on full training data
    random_state=random_seed,
    n_jobs=-1,
    verbose=0,
)
lr_search.fit(x_train, y_train)

print("Best LR params  :", lr_search.best_params_)
print("Best CV PR-AUC  :", round(lr_search.best_score_, 4))

Best LR params  : {'model__solver': 'saga', 'model__penalty': 'l2', 'model__max_iter': 1000, 'model__class_weight': 'balanced', 'model__C': 0.01}
Best CV PR-AUC  : 0.6166


In [30]:
# Tune Random Forest
rf_param_grid = {
    "model__n_estimators"    : [100, 200, 300],
    "model__max_depth"       : [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf" : [1, 2, 4],
    "model__max_features"    : ["sqrt", "log2"],
    "model__class_weight"    : ["balanced"],
    "model__random_state"    : [random_seed],
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=20,
    cv=tscv,
    scoring="average_precision",
    refit=True,
    random_state=random_seed,
    n_jobs=-1,
    verbose=0,
)
rf_search.fit(x_train, y_train)

print("Best RF params  :", rf_search.best_params_)
print("Best CV PR-AUC  :", round(rf_search.best_score_, 4))

Best RF params  : {'model__random_state': 42, 'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 4, 'model__max_features': 'sqrt', 'model__max_depth': None, 'model__class_weight': 'balanced'}
Best CV PR-AUC  : 0.5511


In [31]:
# Tune HistGradientBoosting
hgb_param_grid = {
    "model__learning_rate"    : [0.01, 0.05, 0.1, 0.2],
    "model__max_iter"         : [100, 200, 300, 500],
    "model__max_leaf_nodes"   : [15, 31, 63],
    "model__l2_regularization": [0.0, 0.1, 1.0],
    "model__class_weight"     : ["balanced"],
}

hgb_search = RandomizedSearchCV(
    estimator=hgb_pipeline,
    param_distributions=hgb_param_grid,
    n_iter=20,
    cv=tscv,
    scoring="average_precision",
    refit=True,
    random_state=random_seed,
    n_jobs=-1,
    verbose=0,
)
hgb_search.fit(x_train, y_train)

print("Best HGB params :", hgb_search.best_params_)
print("Best CV PR-AUC  :", round(hgb_search.best_score_, 4))

Best HGB params : {'model__max_leaf_nodes': 15, 'model__max_iter': 200, 'model__learning_rate': 0.01, 'model__l2_regularization': 0.1, 'model__class_weight': 'balanced'}
Best CV PR-AUC  : 0.5332


## Section 13 — Tuned Model Comparison

In [32]:
# Evaluate tuned models on the validation set
# lr_search.best_estimator_ is the pipeline re-fitted with the best params
# on the full training set after the search completes.

lr_tuned_pred  = lr_search.best_estimator_.predict(x_val)
lr_tuned_proba = lr_search.best_estimator_.predict_proba(x_val)[:, 1]
lr_tuned_metrics = evaluate_model("LR_Tuned", y_val, lr_tuned_pred, lr_tuned_proba)

rf_tuned_pred  = rf_search.best_estimator_.predict(x_val)
rf_tuned_proba = rf_search.best_estimator_.predict_proba(x_val)[:, 1]
rf_tuned_metrics = evaluate_model("RF_Tuned", y_val, rf_tuned_pred, rf_tuned_proba)

hgb_tuned_pred  = hgb_search.best_estimator_.predict(x_val)
hgb_tuned_proba = hgb_search.best_estimator_.predict_proba(x_val)[:, 1]
hgb_tuned_metrics = evaluate_model("HGB_Tuned", y_val, hgb_tuned_pred, hgb_tuned_proba)

print("Tuned model validation metrics:")
tuned_df = pd.DataFrame([lr_tuned_metrics, rf_tuned_metrics, hgb_tuned_metrics]).set_index("Model")
display(tuned_df)

Tuned model validation metrics:


,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
Model,,,,,,
LR_Tuned,0.8640,0.2443,0.9143,0.3855,0.9618,0.6005
RF_Tuned,0.9453,0.4400,0.6286,0.5176,0.9584,0.5228
HGB_Tuned,0.9013,0.3030,0.8571,0.4478,0.9558,0.5265


In [33]:
# Before vs After tuning comparison
before_df = comparison_df.loc[["LogisticRegression", "RandomForest", "HistGradientBoosting"]].copy()
after_df  = tuned_df.copy()

print("PR-AUC: Before vs After Tuning")
for orig, tuned in [("LogisticRegression", "LR_Tuned"),
                    ("RandomForest",        "RF_Tuned"),
                    ("HistGradientBoosting","HGB_Tuned")]:
    b = before_df.loc[orig, "PR-AUC"]
    a = after_df.loc[tuned, "PR-AUC"]
    delta = a - b
    print(f"  {orig:<24}: {b:.4f} → {a:.4f}  (Δ {delta:+.4f})")

PR-AUC: Before vs After Tuning
  LogisticRegression      : 0.6294 → 0.6005  (Δ -0.0289)
  RandomForest            : 0.5457 → 0.5228  (Δ -0.0229)
  HistGradientBoosting    : 0.5545 → 0.5265  (Δ -0.0280)


In [34]:
# ROC curves for tuned models
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                              line=dict(dash="dash", color="gray"), name="Random"))

for name, proba in [("LR_Tuned",  lr_tuned_proba),
                    ("RF_Tuned",  rf_tuned_proba),
                    ("HGB_Tuned", hgb_tuned_proba)]:
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc_val = roc_auc_score(y_val, proba)
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"{name} (AUC={auc_val:.3f})"
    ))

fig_roc.update_layout(
    title="ROC Curves — Tuned Models (Validation Set)",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
)
fig_roc.show()
fig_roc.write_json(str(figures_dir / "roc_curves.json"))
fig_roc.write_html(str(figures_dir / "roc_curves.html"))
print("Saved: roc_curves.json & .html")

Saved: roc_curves.json & .html


In [35]:
# Precision-Recall curves
# PR curves are more informative than ROC for imbalanced datasets.
# A model that achieves high precision and high recall is ideal.

fig_pr = go.Figure()
for name, proba in [("LR_Tuned",  lr_tuned_proba),
                    ("RF_Tuned",  rf_tuned_proba),
                    ("HGB_Tuned", hgb_tuned_proba)]:
    prec, rec, _ = precision_recall_curve(y_val, proba)
    pr_auc_val   = average_precision_score(y_val, proba)
    fig_pr.add_trace(go.Scatter(
        x=rec, y=prec, mode="lines",
        name=f"{name} (PR-AUC={pr_auc_val:.3f})"
    ))

# Baseline (random classifier on imbalanced data)
baseline_rate = y_val.mean()
fig_pr.add_trace(go.Scatter(
    x=[0, 1], y=[baseline_rate, baseline_rate],
    mode="lines", line=dict(dash="dash", color="gray"),
    name=f"Random ({baseline_rate:.3f})"
))

fig_pr.update_layout(
    title="Precision-Recall Curves — Tuned Models (Validation Set)",
    xaxis_title="Recall",
    yaxis_title="Precision",
    template="plotly_white",
)
fig_pr.show()
fig_pr.write_json(str(figures_dir / "precision_recall_curves.json"))
fig_pr.write_html(str(figures_dir / "precision_recall_curves.html"))
print("Saved: precision_recall_curves.json & .html")

Saved: precision_recall_curves.json & .html


## Section 14 — Threshold Optimisation

In [36]:
# Threshold sweep on the best tuned model
# We use the HistGradientBoosting tuned model as the primary candidate here.
# The best model selection is confirmed in Section 15.

# Use the tuned HGB probabilities on the validation set
candidate_proba = hgb_tuned_proba

thresholds = np.arange(0.05, 0.75, 0.01)
threshold_results = []

for thresh in thresholds:
    pred_at_thresh = (candidate_proba >= thresh).astype(int)
    threshold_results.append({
        "Threshold": round(thresh, 2),
        "Precision": round(precision_score(y_val, pred_at_thresh, zero_division=0), 4),
        "Recall"   : round(recall_score(y_val, pred_at_thresh, zero_division=0), 4),
        "F1"       : round(f1_score(y_val, pred_at_thresh, zero_division=0), 4),
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.head(20))

,Threshold,Precision,Recall,F1
0,0.05,0.0467,1.0000,0.0892
1,0.06,0.0467,1.0000,0.0893
2,0.07,0.1003,1.0000,0.1823
3,0.08,0.1228,1.0000,0.2188
4,0.09,0.1383,1.0000,0.2431
5,0.10,0.1447,0.9714,0.2519
6,0.11,0.1532,0.9714,0.2646
7,0.12,0.1643,0.9714,0.2810
8,0.13,0.1726,0.9714,0.2931
9,0.14,0.1789,0.9714,0.3022


In [37]:
# Find the threshold that maximises F1
best_f1_row  = threshold_df.loc[threshold_df["F1"].idxmax()]
best_threshold = float(best_f1_row["Threshold"])

print(f"Best F1 threshold : {best_threshold}")
print(f"At that threshold :")
print(f"  Precision : {best_f1_row['Precision']}")
print(f"  Recall    : {best_f1_row['Recall']}")
print(f"  F1        : {best_f1_row['F1']}")

Best F1 threshold : 0.67
At that threshold :
  Precision : 0.4
  Recall    : 0.7429
  F1        : 0.52


In [38]:
# Plotly threshold analysis chart
fig_thresh = go.Figure()
fig_thresh.add_trace(go.Scatter(x=threshold_df["Threshold"], y=threshold_df["Precision"],
                                 mode="lines", name="Precision"))
fig_thresh.add_trace(go.Scatter(x=threshold_df["Threshold"], y=threshold_df["Recall"],
                                 mode="lines", name="Recall"))
fig_thresh.add_trace(go.Scatter(x=threshold_df["Threshold"], y=threshold_df["F1"],
                                 mode="lines", name="F1", line=dict(width=3)))
fig_thresh.add_vline(x=best_threshold, line_dash="dash", line_color="red",
                      annotation_text=f"Best F1 @ {best_threshold}")

fig_thresh.update_layout(
    title="Precision / Recall / F1 vs Classification Threshold (Validation Set)",
    xaxis_title="Threshold",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1.05]),
    template="plotly_white",
)
fig_thresh.show()
fig_thresh.write_json(str(figures_dir / "threshold_analysis.json"))
fig_thresh.write_html(str(figures_dir / "threshold_analysis.html"))
print(f"Saved threshold analysis charts.")
print(f"Selected threshold: {best_threshold}")

Saved threshold analysis charts.
Selected threshold: 0.67


## Section 15 — Final Model Selection

In [39]:
# Final model selection decision
# Compare all three tuned models on the metrics that matter most.

full_comparison = pd.DataFrame([
    lr_tuned_metrics, rf_tuned_metrics, hgb_tuned_metrics
]).set_index("Model")

display(full_comparison[["Recall", "Precision", "F1", "PR-AUC", "ROC-AUC"]])

# Select the model with the highest F1 on the validation set
best_model_name = full_comparison["F1"].idxmax()
print(f"\nSelected model: {best_model_name}  (highest F1 on validation set)")

,Recall,Precision,F1,PR-AUC,ROC-AUC
Model,,,,,
LR_Tuned,0.9143,0.2443,0.3855,0.6005,0.9618
RF_Tuned,0.6286,0.4400,0.5176,0.5228,0.9584
HGB_Tuned,0.8571,0.3030,0.4478,0.5265,0.9558



Selected model: RF_Tuned  (highest F1 on validation set)


In [40]:
# Retrieve the winning pipeline
# Map model name to the search object so we can get its best_estimator_

search_map = {
    "LR_Tuned" : lr_search,
    "RF_Tuned" : rf_search,
    "HGB_Tuned": hgb_search,
}
best_search = search_map[best_model_name]
best_pipeline = best_search.best_estimator_

print(f"Best pipeline retrieved: {best_model_name}")
print(f"Best params: {best_search.best_params_}")

Best pipeline retrieved: RF_Tuned
Best params: {'model__random_state': 42, 'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 4, 'model__max_features': 'sqrt', 'model__max_depth': None, 'model__class_weight': 'balanced'}


In [41]:
# Refit on train + validation combined
# After selecting the model, we merge train and val to give the final model
# as much data as possible before the single test-set evaluation.
# The test set remains untouched at this point.

x_trainval = pd.concat([x_train, x_val], axis=0).reset_index(drop=True)
y_trainval = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

best_pipeline.fit(x_trainval, y_trainval)

print(f"Final model re-fitted on {len(x_trainval):,} samples (train + val).")
print("Test set has not been evaluated yet.")

Final model re-fitted on 4,250 samples (train + val).
Test set has not been evaluated yet.


## Section 16 — Final Test Evaluation

In [42]:
# Final test prediction
test_proba = best_pipeline.predict_proba(x_test)[:, 1]

# Apply the threshold we selected on the validation set
test_pred  = (test_proba >= best_threshold).astype(int)

# Compute all metrics
test_metrics = evaluate_model(f"{best_model_name}_Final", y_test, test_pred, test_proba)

print("Final Test Set Results")
for k, v in test_metrics.items():
    print(f"  {k:<12}: {v}")


Final Test Set Results
  Model       : RF_Tuned_Final
  Accuracy    : 0.9467
  Precision   : 0.5
  Recall      : 0.35
  F1          : 0.4118
  ROC-AUC     : 0.9567
  PR-AUC      : 0.488


In [43]:
# Full classification report
print(classification_report(
    y_test, test_pred,
    target_names=["Not At-Risk (0)", "At-Risk (1)"]
))

                 precision    recall  f1-score   support

Not At-Risk (0)       0.96      0.98      0.97       710
    At-Risk (1)       0.50      0.35      0.41        40

       accuracy                           0.95       750
      macro avg       0.73      0.67      0.69       750
   weighted avg       0.94      0.95      0.94       750



In [44]:
# Final confusion matrix
cm = confusion_matrix(y_test, test_pred)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion Matrix (Test Set)")
print(f"  True Negatives  (TN) : {tn}  — Correctly identified as not at-risk")
print(f"  False Positives (FP) : {fp}  — Flagged as at-risk but actually fine  (unnecessary alert)")
print(f"  False Negatives (FN) : {fn}  — Missed at-risk students (most critical)")
print(f"  True Positives  (TP) : {tp}  — Correctly identified as at-risk")

# Plotly heatmap confusion matrix
cm_fig = px.imshow(
    cm,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=["Predicted: Not At-Risk", "Predicted: At-Risk"],
    y=["Actual: Not At-Risk",    "Actual: At-Risk"],
    text_auto=True,
    color_continuous_scale="Blues",
    title=f"Confusion Matrix — {best_model_name} (Test Set, threshold={best_threshold})",
)
cm_fig.update_layout(template="plotly_white")
cm_fig.show()
cm_fig.write_json(str(figures_dir / "confusion_matrix.json"))
cm_fig.write_html(str(figures_dir / "confusion_matrix.html"))
print("Saved: confusion_matrix.json & .html")

Confusion Matrix (Test Set)
  True Negatives  (TN) : 696  — Correctly identified as not at-risk
  False Positives (FP) : 14  — Flagged as at-risk but actually fine  (unnecessary alert)
  False Negatives (FN) : 26  — Missed at-risk students (most critical)
  True Positives  (TP) : 14  — Correctly identified as at-risk


Saved: confusion_matrix.json & .html


## Section 17 — Error Analysis

In [45]:
# Create an error analysis DataFrame
error_df = df_test[features + [target]].copy()
error_df["predicted"]   = test_pred
error_df["probability"] = test_proba.round(4)
error_df["correct"]     = (error_df["predicted"] == error_df[target]).astype(int)
error_df["error_type"]  = "Correct"
error_df.loc[(error_df[target] == 1) & (error_df["predicted"] == 0), "error_type"] = "False Negative"
error_df.loc[(error_df[target] == 0) & (error_df["predicted"] == 1), "error_type"] = "False Positive"

print("Error type distribution:")
print(error_df["error_type"].value_counts())

Error type distribution:
error_type
Correct           710
False Negative     26
False Positive     14
Name: count, dtype: int64


In [46]:
# Compare False Negatives vs True Positives on key features
# This helps us understand what makes missed at-risk students different.

at_risk_subset = error_df[error_df[target] == 1].copy()
print("Feature comparison: True Positives vs False Negatives")
print("(Among actual at-risk students in the test set)")

key_features = ["attendance_rate", "assignment_completion_rate",
                "average_previous_score", "score_trend", "probability"]

comparison = at_risk_subset.groupby("error_type")[key_features].mean().round(2)
display(comparison)

Feature comparison: True Positives vs False Negatives
(Among actual at-risk students in the test set)


,attendance_rate,assignment_completion_rate,average_previous_score,score_trend,probability
error_type,,,,,
Correct,52.77,35.08,29.15,-4.59,0.76
False Negative,53.66,43.31,37.12,-4.58,0.41


In [47]:
# Scatter: probability distribution for errors
fig_errors = px.strip(
    error_df[error_df[target] == 1],
    x="error_type",
    y="probability",
    color="error_type",
    title="Model Probability — True Positives vs False Negatives",
    labels={"probability": "Predicted Probability", "error_type": "Outcome"},
    stripmode="overlay",
)
fig_errors.add_hline(y=best_threshold, line_dash="dash", line_color="red",
                      annotation_text=f"Threshold = {best_threshold}")
fig_errors.update_layout(template="plotly_white")
fig_errors.show()
fig_errors.write_json(str(figures_dir / "error_analysis.json"))
print("Saved: error_analysis.json")

Saved: error_analysis.json


## Section 18 — Explainability

In [48]:
# Permutation feature importance on the validation set
# We use the validation set so the importance reflects generalisation ability.
# Running this on training data would reflect memorisation, not generalisation.

from sklearn.inspection import permutation_importance

# Apply the final model's preprocessing step first
# (permutation_importance works on raw feature arrays)
x_val_proc_final = best_pipeline.named_steps["preprocessor"].transform(x_val)

perm_result = permutation_importance(
    best_pipeline.named_steps["model"],
    x_val_proc_final,
    y_val,
    n_repeats=20,
    scoring="average_precision",
    random_state=random_seed,
    n_jobs=-1,
)

# The ColumnTransformer re-orders columns: with_missing first, then no_missing
feature_names_ordered = features_with_missing + features_no_missing

importance_df = pd.DataFrame({
    "Feature"       : feature_names_ordered,
    "Importance"    : perm_result.importances_mean.round(4),
    "Std"           : perm_result.importances_std.round(4),
}).sort_values("Importance", ascending=False)

display(importance_df)

,Feature,Importance,Std
5,assignment_completion_rate,0.3179,0.0405
2,late_submission_rate,0.2683,0.0381
10,average_previous_score,0.1781,0.0325
0,attendance_rate,0.1222,0.0213
8,previous_score_2,0.1168,0.0232
6,missing_assignments,0.0913,0.0239
7,previous_score_1,0.0843,0.0273
9,previous_score_3,0.0651,0.0237
1,recent_attendance_rate,0.0633,0.0223
12,score_volatility,0.0627,0.0140


In [49]:
# Feature importance bar chart
fig_importance = px.bar(
    importance_df,
    x="Importance",
    y="Feature",
    orientation="h",
    error_x="Std",
    title=f"Permutation Feature Importance — {best_model_name} (Validation Set)",
    labels={"Importance": "Mean decrease in PR-AUC when feature is shuffled"},
)
fig_importance.update_layout(
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
)
fig_importance.show()
fig_importance.write_json(str(figures_dir / "feature_importance.json"))
fig_importance.write_html(str(figures_dir / "feature_importance.html"))
print("Saved: feature_importance.json & .html")

Saved: feature_importance.json & .html


## Section 19 — Plotly Visualisations

In [50]:
# Target class distribution
target_dist = df["at_risk_next_assessment"].value_counts().reset_index()
target_dist.columns = ["Class", "Count"]
target_dist["Label"] = target_dist["Class"].map({0: "Not At-Risk (0)", 1: "At-Risk (1)"})

fig_dist = px.bar(
    target_dist, x="Label", y="Count",
    text="Count", color="Label",
    title="Target Class Distribution — Full Dataset",
    labels={"Count": "Number of Students"},
    color_discrete_map={"Not At-Risk (0)": "#2196F3", "At-Risk (1)": "#F44336"},
)
fig_dist.update_traces(textposition="outside")
fig_dist.update_layout(template="plotly_white", showlegend=False)
fig_dist.show()
fig_dist.write_json(str(figures_dir / "class_distribution.json"))
print("Saved: class_distribution.json")

Saved: class_distribution.json


In [51]:
# Feature correlation heatmap
corr_matrix = df[features + [target]].corr().round(2)

fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Feature Correlation Heatmap",
    aspect="auto",
)
fig_corr.update_layout(template="plotly_white")
fig_corr.show()
fig_corr.write_json(str(figures_dir / "correlation_heatmap.json"))
print("Saved: correlation_heatmap.json")

Saved: correlation_heatmap.json


In [52]:
# Tuned model performance comparison (grouped bar)
metrics_to_show = ["Recall", "Precision", "F1", "PR-AUC"]
fig_tuned_cmp = go.Figure()
for metric in metrics_to_show:
    fig_tuned_cmp.add_trace(go.Bar(
        name=metric,
        x=tuned_df.index.tolist(),
        y=tuned_df[metric].values,
        text=[f"{v:.3f}" for v in tuned_df[metric].values],
        textposition="outside",
    ))
fig_tuned_cmp.update_layout(
    title="Tuned Model Comparison — Validation Set",
    barmode="group",
    xaxis_title="Model",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1.15]),
    template="plotly_white",
)
fig_tuned_cmp.show()
fig_tuned_cmp.write_json(str(figures_dir / "tuned_model_comparison.json"))
fig_tuned_cmp.write_html(str(figures_dir / "tuned_model_comparison.html"))
print("Saved: tuned_model_comparison.json & .html")

Saved: tuned_model_comparison.json & .html


## Section 20 — Model Serialisation

In [53]:
# Save the final pipeline
# joblib is preferred over pickle for Scikit-learn objects because it
# handles numpy arrays more efficiently.

model_path = artifacts_dir / "model.joblib"
joblib.dump(best_pipeline, model_path)
print(f"Model saved: {model_path}")
print(f"File size  : {model_path.stat().st_size / 1024:.1f} KB")

Model saved: d:\student-performance-risk-prediction\artifacts\model.joblib
File size  : 1535.3 KB


In [54]:
# Save model metadata
# The metadata file documents everything needed to reproduce the result
# and to load the model correctly in a Streamlit app or API.

metadata = {
    "model_name"         : best_model_name,
    "features"           : features,
    "features_with_missing": features_with_missing,
    "target"             : target,
    "target_definition"  : "1 = next_assessment_score < 40, 0 = otherwise",
    "classification_threshold": best_threshold,
    "random_seed"        : random_seed,
    "training_date"      : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_info"       : {
        "file"            : str(data_path),
        "total_rows"      : len(df),
        "train_rows"      : len(x_train),
        "val_rows"        : len(x_val),
        "test_rows"       : len(x_test),
        "at_risk_rate_pct": round(df[target].mean() * 100, 2),
    },
    "best_cv_params"     : best_search.best_params_,
    "best_cv_pr_auc"     : round(best_search.best_score_, 4),
    "validation_metrics" : {
        k: v for k, v in test_metrics.items() if k != "Model"
    },
    "test_metrics"       : {
        k: v for k, v in test_metrics.items() if k != "Model"
    },
}

metadata_path = artifacts_dir / "metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Metadata saved: {metadata_path}")
print("\nMetadata contents:")
print(json.dumps(metadata, indent=2, default=str))

Metadata saved: d:\student-performance-risk-prediction\artifacts\metadata.json

Metadata contents:
{
  "model_name": "RF_Tuned",
  "features": [
    "attendance_rate",
    "recent_attendance_rate",
    "attendance_drop",
    "assignment_completion_rate",
    "missing_assignments",
    "late_submission_rate",
    "days_since_last_submission",
    "previous_score_1",
    "previous_score_2",
    "previous_score_3",
    "average_previous_score",
    "score_trend",
    "score_volatility"
  ],
  "features_with_missing": [
    "attendance_rate",
    "recent_attendance_rate",
    "late_submission_rate",
    "days_since_last_submission",
    "attendance_drop"
  ],
  "target": "at_risk_next_assessment",
  "target_definition": "1 = next_assessment_score < 40, 0 = otherwise",
  "classification_threshold": 0.67,
  "random_seed": 42,
  "training_date": "2026-11-11 13:28:20",
  "dataset_info": {
    "file": "d:\\student-performance-risk-prediction\\data\\generated\\student_performance.csv",
    "tota

In [55]:
# Verify the saved model loads and predicts correctly
loaded_pipeline = joblib.load(model_path)
loaded_proba    = loaded_pipeline.predict_proba(x_test)[:, 1]
loaded_pred     = (loaded_proba >= best_threshold).astype(int)

# Predictions must be identical to the originals
assert (loaded_pred == test_pred).all(), "Mismatch after reload!"
print("Model reload verification passed - predictions match.")
print(f"Loaded model type: {type(loaded_pipeline.named_steps['model']).__name__}")

Model reload verification passed - predictions match.
Loaded model type: RandomForestClassifier


## Section 21 — Saved Artifacts

In [56]:
# List all saved figures
print("Saved Plotly figures:")
for f in sorted(figures_dir.glob("*.json")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} {size_kb:6.1f} KB")

print("\nSaved model artifacts:")
for f in sorted(artifacts_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} {size_kb:6.1f} KB")



Saved Plotly figures:
  assignment_completion_distribution.json         74.4 KB
  attendance_distribution.json                    93.8 KB
  attendance_vs_assignment.json                  160.8 KB
  class_distribution.json                          8.0 KB
  confusion_matrix.json                            7.7 KB
  correlation_heatmap.json                         9.1 KB
  error_analysis.json                              9.2 KB
  feature_importance.json                          8.0 KB
  initial_model_comparison.json                    8.1 KB
  precision_recall_curves.json                    47.9 KB
  roc_curves.json                                 13.5 KB
  threshold_analysis.json                          9.8 KB
  tuned_model_comparison.json                      7.6 KB

Saved model artifacts:
  metadata.json                                    1.6 KB
  model.joblib                                  1535.3 KB
